# 01 — Getting Connected

This notebook checks that you are logged in to OpenShift, grabs an API token, calls the AI models endpoint, and connects to S3-compatible storage (MinIO).

Run each cell from top to bottom by pressing **Shift + Enter**.

## Step 1 — Check you are logged in to OpenShift

In [ ]:
!oc whoami

In [ ]:
!oc project

If either command above shows an error, open a terminal and run `oc login` before continuing.

## Step 2 — Get an API token from your secret

In [ ]:
import subprocess
import base64

SECRET_NAME = "maas-secret"

result = subprocess.run(
    ["oc", "get", "secret", SECRET_NAME, "-o", "jsonpath={.data.token}"],
    capture_output=True, text=True
)

TOKEN = base64.b64decode(result.stdout).decode()
print("Token obtained:", TOKEN[:20], "...")

## Step 3 — List available AI models

In [ ]:
import requests
import json

MAAS_API_URL = "https://maas.apps.ocp.cloud.rhai-tmm.dev"

response = requests.get(
    f"{MAAS_API_URL}/maas-api/v1/models",
    headers={"Authorization": f"Bearer {TOKEN}"},
    verify=False
)

models = response.json()
print(json.dumps(models, indent=2))

## Step 4 — Connect to S3 storage (MinIO)

MinIO is an S3-compatible object store. The default credentials for this environment are **minio / minio1234**.

Update `MINIO_ENDPOINT` below to match your MinIO service URL.

In [ ]:
!pip install boto3 --user -q

In [ ]:
import boto3

MINIO_ENDPOINT   = "http://minio.minio.svc.cluster.local:9000" # ok from within the cluster
MINIO_ACCESS_KEY = "minio" # update this
MINIO_SECRET_KEY = "minio1234" # update this

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-2"
)

# list existing buckets
buckets = s3.list_buckets().get("Buckets", [])
print("Existing buckets:", [b["Name"] for b in buckets])

# create a 'models' bucket if it doesn't exist
existing_names = [b["Name"] for b in buckets]
if "models" not in existing_names:
    s3.create_bucket(Bucket="models")
    print("Created bucket: models")
else:
    print("Bucket 'models' already exists")